In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression

base_dir = Path.cwd()
spectra_path = base_dir / "pattern.xlsx"
label_path = base_dir / "label.xlsx"


spectra_df = pd.read_excel(spectra_path, header=None)
label_df = pd.read_excel(label_path, header=None)
label_df = label_df.iloc[:, -2:].copy()
spectra_df.columns = ["ID"] + [f"Feature_{i}" for i in range(1, spectra_df.shape[1])]
label_df.columns = ["ID", "HA Yield"]

id_col_spectra = spectra_df.columns[0]
id_col_label = label_df.columns[0]
target_col = label_df.columns[1]

def normalize_id(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return str(x)
    if isinstance(x, (float, np.floating)):
        return str(int(x)) if x.is_integer() else str(x).rstrip("0").rstrip(".")
    return str(x).strip()

spectra_df[id_col_spectra] = spectra_df[id_col_spectra].map(normalize_id)
label_df[id_col_label] = label_df[id_col_label].map(normalize_id)

label_df = label_df[[id_col_label, target_col]].copy()
label_df = label_df.rename(columns={id_col_label: id_col_spectra})

label_check = label_df.groupby(id_col_spectra)[target_col].nunique(dropna=False)
conflict_ids = label_check[label_check > 1].index.tolist()
if conflict_ids:
    raise ValueError(f"The following IDs have multiple different HA yields, please check label.xlsx: {conflict_ids}")

label_unique = label_df.drop_duplicates(subset=[id_col_spectra], keep="first")
df = spectra_df.merge(label_unique, on=id_col_spectra, how="left")

if df[target_col].isna().any():
    missing_ids = df.loc[df[target_col].isna(), id_col_spectra].unique()
    raise ValueError(f"The following IDs are not found in label.xlsx: {missing_ids}")


numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols if col != target_col]
X = df[feature_cols].astype(float)
y = df[target_col].astype(float)



X_standard = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0).replace(0, 1)


n_components = 4
max_components = min(X_standard.shape[0] - 1, X_standard.shape[1])
if n_components > max_components:
    raise ValueError(f"At most {max_components} PLS components can be extracted from the current data, cannot set to {n_components}.")

X_pls_input = X_standard.values
y_pls_input = y.values.reshape(-1, 1)

pls = PLSRegression(n_components=n_components, scale=False)
pls.fit(X_pls_input, y_pls_input)


X_scores = pls.x_scores_
X_loadings = pls.x_loadings_


X_reconstructed = X_scores @ X_loadings.T
ss_total = np.sum(X_pls_input ** 2)
ss_residual = np.sum((X_pls_input - X_reconstructed) ** 2)
cumulative_variance_ratio = 1 - ss_residual / ss_total


single_var_ratio = []
step_cum_var_ratio = []
cum_temp = 0.0

for comp_idx in range(1, n_components + 1):
    
    X_rec_single = X_scores[:, :comp_idx] @ X_loadings[:, :comp_idx].T
    ss_res_comp = np.sum((X_pls_input - X_rec_single) ** 2)
    comp_cum_ratio = 1 - ss_res_comp / ss_total
    
    single_ratio = comp_cum_ratio - cum_temp
    single_var_ratio.append(single_ratio)
    step_cum_var_ratio.append(comp_cum_ratio)
    cum_temp = comp_cum_ratio


pls_variance_df = pd.DataFrame({
    "Component": [f"PLS_Component_{i+1}" for i in range(n_components)],
    "Single_Variance_Ratio": single_var_ratio,
    "Cumulative_Variance_Ratio": step_cum_var_ratio
})


pls_cols = [f"PLS_Component_{i + 1}" for i in range(n_components)]
df_pls = pd.DataFrame(X_scores, columns=pls_cols)
df_pls.insert(0, id_col_spectra, df[id_col_spectra].values)
df_pls[target_col] = y.values


X_model = df_pls[pls_cols].astype(float)
y_model = df_pls[target_col].astype(float)
groups = df_pls[id_col_spectra].astype(str)



print("="*50)
print("Individual and Stepwise Cumulative Variance Explained by PLS Components:")
print(pls_variance_df.round(4))
print("="*50)
print(f"All {n_components} components total cumulative variance explained:{cumulative_variance_ratio:.4f}")
print("PLS Sample Score Matrix Shape (Samples x Components):", X_scores.shape)
print("Reduced Feature Columns:", pls_cols)

Individual and Stepwise Cumulative Variance Explained by PLS Components:
         Component  Single_Variance_Ratio  Cumulative_Variance_Ratio
0  PLS_Component_1                 0.1896                     0.1896
1  PLS_Component_2                 0.7580                     0.9476
2  PLS_Component_3                 0.0230                     0.9706
3  PLS_Component_4                 0.0015                     0.9721
All 4 components total cumulative variance explained:0.9721
PLS Sample Score Matrix Shape (Samples x Components): (148, 4)
Reduced Feature Columns: ['PLS_Component_1', 'PLS_Component_2', 'PLS_Component_3', 'PLS_Component_4']
